# 04 — Calibration-only time-series EDA

This notebook studies model-visible behaviour using **only the calibration
partition**. It does not mount or inspect faults, tickets or holdout truth.

The goal is to choose defensible reference behaviour—not to search for a model
that happens to fit labels. Plots retain time order, long gaps are not filled,
and seasonality is selected only when several cycles repeat.


## 1. Setup and calibration guard


In [ ]:
from pathlib import Path
import os
import sys


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import hashlib

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL

from telco_anomaly.io import (
    immutable_output_directory, load_config, read_json, resolve_data_root,
    write_json,
)

sns.set_theme(style="whitegrid")
DATA_ROOT = resolve_data_root()
DATASET = os.getenv("TELCO_DATASET", "synthetic_pon")
default_core_runs = {
    "synthetic_pon": "synthetic_pon_core_v2",
    "ran_pm": "ran_pm_v1",
    "microsoft_optical": "microsoft_optical_v1",
}
if DATASET not in default_core_runs:
    raise ValueError(f"Calibration EDA is not configured for {DATASET!r}")
legacy_core = os.getenv("PON_CORE_RUN_ID") if DATASET == "synthetic_pon" else None
legacy_eda = os.getenv("PON_EDA_RUN_ID") if DATASET == "synthetic_pon" else None
CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", legacy_core or default_core_runs[DATASET]
)
EDA_RUN_ID = os.getenv(
    "TELCO_EDA_RUN_ID", legacy_eda or f"{DATASET}_calibration_eda_v2"
)
RUN_ROOT = DATA_ROOT / "core" / DATASET / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
OUTPUT_ROOT = DATA_ROOT / "eda" / DATASET / EDA_RUN_ID
ENTITY_LIMIT = int(os.getenv("PON_EDA_ENTITY_LIMIT", "12"))
MAX_PLOT_POINTS = int(os.getenv("PON_EDA_MAX_PLOT_POINTS", "3000"))

if not CORE_ROOT.is_dir():
    raise FileNotFoundError("Run Notebook 02 first")
if (RUN_ROOT / "SPEC-EVAL").exists():
    raise PermissionError("EDA must use the truth-unmounted canonical run")

core_manifest = read_json(CORE_ROOT / "manifest.json")
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
topology_config = load_config("topology", project_root=PROJECT_ROOT)
partitions = pd.read_parquet(RUN_ROOT / "SPLITS" / "time_partitions.parquet")
calibration = partitions.loc[partitions["partition"].eq("calibration")].iloc[0]
CALIBRATION_START = pd.to_datetime(calibration["start_ts"], utc=True)
CALIBRATION_END = pd.to_datetime(calibration["end_ts"], utc=True)

display(pd.Series({
    "core": str(CORE_ROOT),
    "dataset": DATASET,
    "calibration_start": CALIBRATION_START,
    "calibration_end_exclusive": CALIBRATION_END,
    "entity_sample": ENTITY_LIMIT,
    "truth_mounted": False,
}, name="value").to_frame())


## 2. Query calibration telemetry without materialising the full panel


In [ ]:
telemetry_glob = str(CORE_ROOT / "telemetry" / "part-*.parquet").replace("'", "''")
connection = duckdb.connect()
connection.execute("SET threads = 2")
connection.execute("SET preserve_insertion_order = false")
connection.register("metric_catalogue", catalogue)
connection.execute(f"""
    CREATE VIEW calibration_telemetry AS
    SELECT *
    FROM read_parquet('{telemetry_glob}')
    WHERE event_ts >= TIMESTAMPTZ '{CALIBRATION_START.isoformat()}'
      AND event_ts <  TIMESTAMPTZ '{CALIBRATION_END.isoformat()}'
""")

overview = connection.execute("""
    SELECT count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           count(DISTINCT metric_id) AS metrics,
           min(event_ts) AS observed_from,
           max(event_ts) AS observed_to,
           sum(quality_code = 'invalid') / count(*)::DOUBLE AS invalid_rate,
           sum(quality_code = 'clipped') / count(*)::DOUBLE AS clipped_rate
    FROM calibration_telemetry
""").df()
display(overview.T.rename(columns={0: "value"}))


## 3. Metric-level robust profile


In [ ]:
metric_profiles = connection.execute("""
    SELECT metric_id,
           count(*) AS rows,
           count(value) FILTER (quality_code <> 'invalid') AS valid_rows,
           avg(CASE WHEN quality_code = 'invalid' THEN 1.0 ELSE 0.0 END) AS invalid_rate,
           avg(CASE WHEN quality_code = 'clipped' THEN 1.0 ELSE 0.0 END) AS clipped_rate,
           avg(CASE WHEN value = 0 THEN 1.0 ELSE 0.0 END)
               FILTER (quality_code <> 'invalid') AS zero_rate,
           approx_quantile(value, 0.05) FILTER (quality_code <> 'invalid') AS q05,
           approx_quantile(value, 0.50) FILTER (quality_code <> 'invalid') AS median,
           approx_quantile(value, 0.95) FILTER (quality_code <> 'invalid') AS q95
    FROM calibration_telemetry
    GROUP BY metric_id
    ORDER BY metric_id
""").df().merge(catalogue, on="metric_id", how="left", validate="one_to_one")
display(metric_profiles[[
    "metric_id", "measurement_kind", "unit", "rows", "valid_rows",
    "invalid_rate", "clipped_rate", "zero_rate", "q05", "median", "q95",
]])


## 4. Deterministic entity sample and per-series coverage

The sample is selected by a stable hash, not by anomaly score or labels. This
keeps the detailed plots reproducible while the metric-level profile above
still uses the full calibration partition.


In [ ]:
def stable_key(value):
    return hashlib.sha256(str(value).encode()).hexdigest()


entities = sorted(registry["entity_id"].astype(str), key=stable_key)
selected_entities = entities[:min(ENTITY_LIMIT, len(entities))]
connection.register("selected_entities", pd.DataFrame({"entity_id": selected_entities}))

sample = connection.execute("""
    SELECT t.event_ts, t.entity_id, t.episode_id, t.metric_id,
           t.value, t.quality_code
    FROM calibration_telemetry AS t
    JOIN selected_entities AS s USING (entity_id)
    ORDER BY entity_id, metric_id, event_ts
""").df()
sample["event_ts"] = pd.to_datetime(sample["event_ts"], utc=True)

cadence_by_metric = catalogue.set_index("metric_id")["expected_cadence_seconds"]
series_profiles = (
    sample.groupby(["entity_id", "episode_id", "metric_id"], as_index=False)
    .agg(
        observed_rows=("event_ts", "size"),
        valid_rows=("value", "count"),
        invalid_rate=("quality_code", lambda x: x.eq("invalid").mean()),
        clipped_rate=("quality_code", lambda x: x.eq("clipped").mean()),
    )
)
validity = registry[["entity_id", "valid_from", "valid_to"]].copy()
validity["valid_from"] = pd.to_datetime(validity["valid_from"], utc=True)
validity["valid_to"] = pd.to_datetime(validity["valid_to"], utc=True)
validity["coverage_start"] = validity["valid_from"].fillna(CALIBRATION_START).clip(
    lower=CALIBRATION_START
)
validity["coverage_end"] = validity["valid_to"].fillna(CALIBRATION_END).clip(
    upper=CALIBRATION_END
)
series_profiles = series_profiles.merge(
    validity[["entity_id", "coverage_start", "coverage_end"]],
    on="entity_id", how="left", validate="many_to_one",
)
active_seconds = (
    series_profiles["coverage_end"] - series_profiles["coverage_start"]
).dt.total_seconds().clip(lower=0)
series_profiles["expected_rows"] = np.ceil(
    active_seconds / series_profiles["metric_id"].map(cadence_by_metric)
)
series_profiles["timestamp_coverage"] = (
    series_profiles["observed_rows"] / series_profiles["expected_rows"]
).clip(upper=1)
display(series_profiles.head(20))


## 5. Coverage and distribution plots


In [ ]:
coverage = series_profiles.pivot(
    index="entity_id", columns="metric_id", values="timestamp_coverage"
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(coverage, vmin=0, vmax=1, cmap="viridis", ax=axes[0])
axes[0].set_title("Calibration timestamp coverage")

valid_sample = sample.loc[sample["quality_code"].ne("invalid")].dropna(subset=["value"])
plot_sample = pd.concat([
    frame.sample(min(len(frame), 4000), random_state=42)
    for _, frame in valid_sample.groupby("metric_id")
], ignore_index=True)
centre = metric_profiles.set_index("metric_id")["median"]
spread = (
    metric_profiles.set_index("metric_id")["q95"]
    - metric_profiles.set_index("metric_id")["q05"]
).replace(0, np.nan)
plot_sample["robust_position"] = (
    plot_sample["value"] - plot_sample["metric_id"].map(centre)
) / plot_sample["metric_id"].map(spread)
plot_sample["robust_position"] = plot_sample["robust_position"].clip(-3, 3)
sns.boxplot(
    data=plot_sample, x="metric_id", y="robust_position",
    showfliers=False, ax=axes[1], color="#72B7B2",
)
axes[1].tick_params(axis="x", rotation=70)
axes[1].set_title("Robustly normalised distributions (sampled)")
axes[1].set_ylabel("(value − median) / (q95 − q05)")
plt.tight_layout()
plt.show()


## 6. Time-series views with rolling robust statistics

One high-coverage series is selected per measurement kind. Lines are
downsampled only for display; rolling medians and interquartile ranges are
computed before display sampling.


In [ ]:
catalogue_index = catalogue.set_index("metric_id")
representatives = (
    series_profiles.sort_values(
        ["metric_id", "timestamp_coverage", "valid_rows"],
        ascending=[True, False, False],
    ).groupby("metric_id", as_index=False).first()
)
configured_focus = (
    load_config("metric_registry", project_root=PROJECT_ROOT)["eda_focus_metrics"]
    if DATASET == "synthetic_pon" else catalogue["metric_id"].tolist()
)
focus_metrics = [
    metric_id for metric_id in configured_focus
    if metric_id in set(representatives["metric_id"])
][:8]
representatives = representatives.loc[
    representatives["metric_id"].isin(focus_metrics)
].set_index("metric_id").loc[focus_metrics].reset_index()


def one_series(entity_id, episode_id, metric_id):
    rows = sample.loc[
        sample["entity_id"].astype(str).eq(str(entity_id))
        & sample["episode_id"].astype(str).eq(str(episode_id))
        & sample["metric_id"].eq(metric_id)
        & sample["quality_code"].ne("invalid"),
        ["event_ts", "value"],
    ].dropna().drop_duplicates("event_ts").sort_values("event_ts")
    return rows.set_index("event_ts")["value"]


for row in representatives.itertuples(index=False):
    series = one_series(row.entity_id, row.episode_id, row.metric_id)
    cadence = float(catalogue_index.loc[row.metric_id, "expected_cadence_seconds"])
    rolling_points = max(4, round(24 * 3600 / cadence))
    rolling = series.rolling(rolling_points, min_periods=max(3, rolling_points // 4))
    step = max(1, len(series) // MAX_PLOT_POINTS)

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(series.index[::step], series.iloc[::step], alpha=0.45, label="observed")
    ax.plot(rolling.median().index[::step], rolling.median().iloc[::step],
            color="black", label="24-hour rolling median")
    ax.fill_between(
        series.index[::step], rolling.quantile(0.25).iloc[::step],
        rolling.quantile(0.75).iloc[::step], alpha=0.20, label="rolling IQR",
    )
    ax.set_title(f"{row.metric_id} — {row.entity_id}")
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


## 7. Cadence and dependence

Autocorrelation is computed on a regular, contiguous series. Long gaps split a
series and are never interpolated. Cross-metric dependence is calculated on
within-entity changes, which reduces spurious correlation caused by different
entity baselines.


In [ ]:
def longest_regular_segment(series, cadence_seconds):
    if series.empty:
        return series
    cadence = pd.Timedelta(seconds=float(cadence_seconds))
    segment = series.index.to_series().diff().gt(cadence * 1.5).cumsum()
    longest_id = segment.value_counts().idxmax()
    return series.loc[segment.eq(longest_id).to_numpy()].asfreq(cadence)


def fill_short_internal_gaps(series, limit=2):
    """Fill only short internal gaps; preserve longer missing stretches."""
    filled = series.interpolate(limit=limit, limit_area="inside")
    valid_runs = filled.notna().ne(filled.notna().shift()).cumsum()
    valid = filled.notna()
    if not valid.any():
        return filled.iloc[:0]
    longest_id = valid_runs.loc[valid].value_counts().idxmax()
    return filled.loc[valid_runs.eq(longest_id)].dropna()


acf_rows = []
for row in representatives.itertuples(index=False):
    cadence = catalogue_index.loc[row.metric_id, "expected_cadence_seconds"]
    series = fill_short_internal_gaps(longest_regular_segment(
        one_series(row.entity_id, row.episode_id, row.metric_id), cadence
    ))
    acf_rows.append({
        "metric_id": row.metric_id,
        "entity_id": row.entity_id,
        "regular_observations": len(series),
        "lag_1_correlation": series.autocorr(1) if len(series) >= 3 else np.nan,
    })
    if len(series) >= 200 and series.nunique() > 2:
        fig, ax = plt.subplots(figsize=(10, 3.5))
        plot_acf(series.iloc[:10000], lags=min(96, len(series) // 4), zero=False, ax=ax)
        ax.set_title(f"ACF — {row.metric_id} / {row.entity_id}")
        plt.tight_layout()
        plt.show()

acf_summary = pd.DataFrame(acf_rows)
display(acf_summary)

pivot = valid_sample.pivot_table(
    index=["entity_id", "event_ts"], columns="metric_id", values="value", aggfunc="first"
).sort_index()
changes = pivot.groupby(level="entity_id").diff()
change_correlation = changes.corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(change_correlation, vmin=-1, vmax=1, center=0, cmap="vlag", ax=ax)
ax.set_title("Spearman correlation of within-ONT changes")
plt.tight_layout()
plt.show()


## 8. Test candidate seasonality; do not assume it

For each declared candidate metric, repeated daily and weekly cycles are
checked on representative series. A seasonal reference is recommended only
when at least six complete cycles exist and the cycle shapes agree. These are
pre-model calibration diagnostics, not label-driven tuning.


In [ ]:
MIN_CYCLES = 6
MIN_SEASONAL_SERIES = 5
MIN_REPEATABILITY = 0.50
MIN_APPROVAL_SHARE = 0.60
periods = {"daily": 24 * 3600, "weekly": 7 * 24 * 3600}


def cycle_repeatability(series, period_points):
    values = series.to_numpy(dtype=float)
    cycles = len(values) // period_points
    if cycles < MIN_CYCLES:
        return np.nan, cycles
    matrix = values[:cycles * period_points].reshape(cycles, period_points)
    template = np.nanmedian(matrix, axis=0)
    correlations = []
    for cycle in matrix:
        pairs = pd.DataFrame({"cycle": cycle, "template": template}).dropna()
        correlations.append(
            pairs["cycle"].corr(pairs["template"], method="spearman")
            if len(pairs) >= period_points * 0.80 else np.nan
        )
    return float(np.nanmedian(correlations)), cycles


seasonality_rows = []
seasonal_metrics = catalogue.loc[catalogue["seasonality_candidate"], "metric_id"]
for metric_id in seasonal_metrics:
    candidates = series_profiles.loc[
        series_profiles["metric_id"].eq(metric_id)
    ].sort_values(["timestamp_coverage", "valid_rows"], ascending=False).head(8)
    cadence = float(catalogue_index.loc[metric_id, "expected_cadence_seconds"])
    for candidate in candidates.itertuples(index=False):
        regular = fill_short_internal_gaps(longest_regular_segment(
            one_series(candidate.entity_id, candidate.episode_id, metric_id), cadence
        ))
        for period_name, period_seconds in periods.items():
            repeatability, cycles = cycle_repeatability(
                regular, round(period_seconds / cadence)
            )
            seasonality_rows.append({
                "metric_id": metric_id,
                "entity_id": candidate.entity_id,
                "episode_id": candidate.episode_id,
                "period": period_name,
                "period_seconds": period_seconds,
                "cycles": cycles,
                "repeatability": repeatability,
            })

seasonality_evidence = pd.DataFrame(seasonality_rows, columns=[
    "metric_id", "entity_id", "episode_id", "period", "period_seconds",
    "cycles", "repeatability",
])
decision_rows = []
for (metric_id, period_name), group in seasonality_evidence.groupby(
    ["metric_id", "period"], sort=True
):
    usable = group.loc[
        group["cycles"].ge(MIN_CYCLES) & group["repeatability"].notna()
    ]
    repeatability = usable["repeatability"].median() if len(usable) else np.nan
    approval_share = (
        usable["repeatability"].ge(MIN_REPEATABILITY).mean()
        if len(usable) else np.nan
    )
    approved = bool(
        len(usable) >= MIN_SEASONAL_SERIES
        and repeatability >= MIN_REPEATABILITY
        and approval_share >= MIN_APPROVAL_SHARE
    )
    reason = (
        "insufficient_series" if len(usable) < MIN_SEASONAL_SERIES
        else "weak_or_inconsistent_pattern" if not approved
        else "repeatable_across_entities"
    )
    decision_rows.append({
        "metric_id": metric_id,
        "period": period_name,
        "period_seconds": int(group["period_seconds"].iloc[0]),
        "series_assessed": len(group),
        "usable_series": len(usable),
        "median_repeatability": repeatability,
        "approval_share": approval_share,
        "approved": approved,
        "reason": reason,
    })

seasonality_decisions = pd.DataFrame(decision_rows, columns=[
    "metric_id", "period", "period_seconds", "series_assessed",
    "usable_series", "median_repeatability", "approval_share", "approved",
    "reason",
])
display(seasonality_decisions)

best = seasonality_decisions.loc[
    seasonality_decisions["approved"]
].sort_values("median_repeatability", ascending=False)
if len(best):
    row = best.iloc[0]
    representative = seasonality_evidence.loc[
        seasonality_evidence["metric_id"].eq(row["metric_id"])
        & seasonality_evidence["period"].eq(row["period"])
    ].sort_values("repeatability", ascending=False).iloc[0]
    cadence = float(catalogue_index.loc[row["metric_id"], "expected_cadence_seconds"])
    regular = fill_short_internal_gaps(longest_regular_segment(
        one_series(representative.entity_id, representative.episode_id, row["metric_id"]),
        cadence,
    ))
    period_points = round(row["period_seconds"] / cadence)
    result = STL(regular, period=period_points, robust=True).fit()
    figure = result.plot()
    figure.set_size_inches(12, 8)
    figure.suptitle(f"Robust STL — {row['metric_id']} ({row['period']})", y=1.01)
    plt.show()


## 9. Peer availability and an example peer-correlation matrix


In [ ]:
topology_path = CORE_ROOT / "topology_memberships.parquet"
if topology_path.exists():
    memberships = pd.read_parquet(topology_path)
    group_sizes = (
        memberships.groupby(["group_type", "group_id"], as_index=False)
        .agg(entities=("entity_id", "nunique"))
    )
    peer_availability = (
        group_sizes.groupby("group_type", as_index=False)["entities"]
        .agg(groups="count", minimum="min", median="median", maximum="max")
    )
    display(peer_availability)

    physical = memberships.loc[
        memberships["group_family"].eq("physical_topology")
    ]
    group_counts = (
        physical.groupby(["group_type", "group_id"])["entity_id"].nunique()
    )
    preferred_levels = topology_config["peer_policy"]["preferred_levels"]
    eligible = group_counts.loc[group_counts.ge(2)]
    if len(eligible):
        available_levels = set(eligible.index.get_level_values(0))
        ordered_levels = [level for level in preferred_levels if level in available_levels]
        ordered_levels += sorted(available_levels - set(ordered_levels))
        group_type = ordered_levels[0]
        group_id = eligible.loc[group_type].sort_values(ascending=False).index[0]
        peer_entities = physical.loc[
            physical["group_type"].eq(group_type)
            & physical["group_id"].astype(str).eq(str(group_id)), "entity_id"
        ].astype(str).unique().tolist()
        peer_metric = catalogue.loc[catalogue["peer_eligible"], "metric_id"].iloc[0]
        connection.register("peer_entities", pd.DataFrame({"entity_id": peer_entities}))
        peer_data = connection.execute("""
            SELECT t.event_ts, t.entity_id, t.value
            FROM calibration_telemetry AS t
            JOIN peer_entities AS p USING (entity_id)
            WHERE t.metric_id = ? AND t.quality_code <> 'invalid'
            ORDER BY t.event_ts, t.entity_id
        """, [peer_metric]).df()
        peer_matrix = peer_data.pivot(index="event_ts", columns="entity_id", values="value")
        peer_correlation = peer_matrix.diff().corr(method="spearman")

        if 1 < len(peer_correlation) <= 40:
            fig, ax = plt.subplots(figsize=(9, 7))
            sns.heatmap(peer_correlation, vmin=-1, vmax=1, center=0, cmap="vlag", ax=ax)
            ax.set_title(f"Peer change correlation — {group_type} {group_id}")
            plt.tight_layout()
            plt.show()
    else:
        peer_correlation = pd.DataFrame()
        print("No topology group has at least two observable entities")
else:
    peer_availability = pd.DataFrame()
    peer_correlation = pd.DataFrame()
    print("Topology unavailable: peer-relative channels must remain disabled")


## 10. Freeze calibration evidence for Notebook 05


In [ ]:
base_cadence_seconds = float(
    pd.to_numeric(catalogue["expected_cadence_seconds"], errors="coerce").median()
)
approved_periods = seasonality_decisions.loc[
    seasonality_decisions["approved"], "period_seconds"
].tolist()
largest_approved_period = max(approved_periods, default=24 * 3600)
calibration_span_seconds = (CALIBRATION_END - CALIBRATION_START).total_seconds()
history_window_seconds = min(
    calibration_span_seconds / 3,
    max(14 * 24 * 3600, 4 * largest_approved_period),
)
minimum_history_seconds = min(
    history_window_seconds / 2,
    max(2 * 24 * 3600, 2 * largest_approved_period),
)
dispersion_window_seconds = min(
    24 * 3600,
    max(6 * 3600, history_window_seconds / 14),
)

seasonality_by_metric = {
    metric_id: (
        int(group.loc[group["approved"]]
            .sort_values("median_repeatability", ascending=False)
            .iloc[0]["period_seconds"])
        if group["approved"].any() else None
    )
    for metric_id, group in seasonality_decisions.groupby("metric_id")
}
for metric_id in catalogue["metric_id"]:
    seasonality_by_metric.setdefault(metric_id, None)

eda_decisions = {
    "canonical_fingerprint": core_manifest["fingerprint"],
    "base_cadence_seconds": base_cadence_seconds,
    "history_window_seconds": int(history_window_seconds),
    "minimum_history_seconds": int(minimum_history_seconds),
    "dispersion_window_seconds": int(dispersion_window_seconds),
    "seasonality_decisions": seasonality_by_metric,
    "decision_basis": {
        "partition": "calibration_only",
        "minimum_complete_cycles": MIN_CYCLES,
        "minimum_series": MIN_SEASONAL_SERIES,
        "minimum_repeatability": MIN_REPEATABILITY,
        "minimum_approval_share": MIN_APPROVAL_SHARE,
        "long_gaps_filled": False,
        "maximum_short_gap_fill_observations": 2,
    },
}

outputs = {
    "metric_profiles": metric_profiles,
    "series_profiles": series_profiles,
    "acf_summary": acf_summary,
    "seasonality_decisions": seasonality_decisions,
    "seasonality_evidence": seasonality_evidence,
    "peer_availability": peer_availability,
    "change_correlation": change_correlation.reset_index(),
}
eda_manifest = {
    "dataset": DATASET,
    "partition": "calibration",
    "calibration_start": CALIBRATION_START,
    "calibration_end_exclusive": CALIBRATION_END,
    "core_fingerprint": core_manifest["fingerprint"],
    "truth_files_read": [],
    "selected_entities": selected_entities,
    "output_rows": {name: len(frame) for name, frame in outputs.items()},
}

if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "eda_manifest.json")
    assert previous["core_fingerprint"] == core_manifest["fingerprint"]
    print("Using existing immutable EDA output:", OUTPUT_ROOT)
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        for name, frame in outputs.items():
            frame.to_parquet(output / f"{name}.parquet", index=False)
        write_json(output / "eda_decisions.json", eda_decisions)
        write_json(output / "eda_manifest.json", eda_manifest)
    print("Saved calibration evidence:", OUTPUT_ROOT)

assert eda_manifest["truth_files_read"] == []
display(pd.Series(eda_decisions, name="decision").to_frame())
connection.close()
print("PASS — EDA used calibration telemetry only")
print("Next: 05_FEATURE_ENGINEERING.ipynb")
